# Run_BrickScanner_Git

Purpose: Scan only changed files from GitHub branch or recent commits.

Retrieves GitHub PAT from Databricks secret scope, queries GitHub API for changed files,
converts repo-relative paths to workspace paths, applies exclusion rules, and executes scan.

**Widgets:**
- `repo_full_path`: Full repository path (org/repo format)
- `branch_name`: GitHub branch name to scan

## Define Widgets and Setup

In [0]:
import sys
import subprocess
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("[Run_BrickScanner_Git]")

# Define widgets
dbutils.widgets.text("repo_full_path", "hcsc-core/data-pipeline", "Repository (org/repo)")
dbutils.widgets.text("branch_name", "main", "Branch Name")

repo_full_path = dbutils.widgets.get("repo_full_path")
branch_name = dbutils.widgets.get("branch_name")

## Install Dependencies

In [0]:
print("[Git] Installing dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "/dbfs/user/sai_kalluri@bcbsil.com/CLOUD_BIS_AI_POC/BrickScanner/requirements.txt"])
dbutils.library.restartPython()

## Import and Retrieve Credentials

In [0]:
import requests
import json
from brickscanner.config import GITHUB_SECRET_SCOPE, GITHUB_SECRET_KEY, GITHUB_ORG, GITHUB_COMMITS_LIMIT, EXCLUSION_LIST

# Retrieve GitHub PAT from secret scope
try:
    github_pat = dbutils.secrets.get(scope=GITHUB_SECRET_SCOPE, key=GITHUB_SECRET_KEY)
except Exception as e:
    logger.error(f"Failed to retrieve GitHub PAT from secrets: {e}")
    raise ValueError(f"Configure secret {GITHUB_SECRET_SCOPE}/{GITHUB_SECRET_KEY}")

## GitHub API Functions

In [0]:
def get_changed_files(repo_full_path, branch_name, github_pat, noof_commits=1):
    """
    Query GitHub API for changed files in recent commits.
    
    Args:
        repo_full_path: GitHub repo in org/repo format
        branch_name: Branch name
        github_pat: GitHub Personal Access Token
        noof_commits: Number of recent commits to check (default 1)
    
    Returns:
        Set of changed filenames (repo-relative paths)
    """
    headers = {
        "Authorization": f"token {github_pat}",
        "Accept": "application/vnd.github.v3+json"
    }
    
    # Get commits
    url = f"https://api.github.com/repos/{repo_full_path}/commits?sha={branch_name}&per_page={noof_commits}"
    logger.info(f"[Git] Querying: {url}")
    
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    commits = response.json()
    
    if not commits:
        logger.warning("[Git] No commits found")
        return set()
    
    changed_files = set()
    allowed_extensions = {".ipynb", ".py", ".sql"}
    
    for commit in commits:
        commit_sha = commit["sha"]
        commit_url = f"https://api.github.com/repos/{repo_full_path}/commits/{commit_sha}"
        
        logger.info(f"[Git] Fetching files from commit: {commit_sha}")
        commit_response = requests.get(commit_url, headers=headers)
        commit_response.raise_for_status()
        commit_data = commit_response.json()
        
        for file_obj in commit_data.get("files", []):
            filename = file_obj["filename"]
            # Check extension
            if any(filename.endswith(ext) for ext in allowed_extensions):
                changed_files.add(filename)
    
    logger.info(f"[Git] Found {len(changed_files)} changed files")
    return changed_files

## Path Conversion Functions

In [0]:
def convert_to_workspace_paths(repo_relative_paths, workspace_base="/Workspace"):
    """
    Convert GitHub repo-relative paths to Databricks workspace paths.
    
    Args:
        repo_relative_paths: Set of repo-relative paths
        workspace_base: Base workspace path
    
    Returns:
        List of workspace paths, with .ipynb suffix stripped for notebooks
    """
    workspace_paths = []
    for path in repo_relative_paths:
        # Strip .ipynb if present (notebooks in workspace don't have suffix)
        if path.endswith(".ipynb"):
            path = path[:-6]
        
        workspace_path = f"{workspace_base}/{path}"
        workspace_paths.append(workspace_path)
    
    return workspace_paths


def apply_exclusions(paths, exclusion_list):
    """
    Filter paths by exclusion list (suffix-matched).
    
    Args:
        paths: List of paths
        exclusion_list: List of suffix patterns to exclude
    
    Returns:
        Tuple of (included_paths, excluded_paths)
    """
    included = []
    excluded = []
    
    for path in paths:
        if any(path.endswith(suffix) for suffix in exclusion_list):
            excluded.append(path)
        else:
            included.append(path)
    
    return included, excluded

## Main Execution

In [0]:
# Main execution
try:
    print(f"[Git] Scanning repository: {repo_full_path}")
    print(f"[Git] Branch: {branch_name}")
    
    # Get changed files from GitHub
    changed_files = get_changed_files(repo_full_path, branch_name, github_pat, GITHUB_COMMITS_LIMIT)
    
    if not changed_files:
        print("[Git] No changed files found, exiting")
        dbutils.notebook.exit("No changes")
    
    print(f"[Git] Found {len(changed_files)} changed files")
    
    # Convert to workspace paths
    workspace_paths = convert_to_workspace_paths(changed_files)
    
    # Apply exclusions
    included_paths, excluded_paths = apply_exclusions(workspace_paths, EXCLUSION_LIST)
    
    print(f"[Git] Included paths: {len(included_paths)}")
    for p in included_paths:
        print(f"      {p}")
    
    if excluded_paths:
        print(f"[Git] Excluded paths: {len(excluded_paths)}")
        for p in excluded_paths:
            print(f"      {p}")
    
    if not included_paths:
        print("[Git] All files excluded, exiting")
        dbutils.notebook.exit("All excluded")
    
    # Invoke Run_BrickScanner with included paths
    print("[Git] Invoking Run_BrickScanner...")
    include_paths_str = ",".join(included_paths)
    
    dbutils.notebook.run(
        "./Run_BrickScanner.ipynb",
        timeout_seconds=3600,
        arguments={
            "include_paths": include_paths_str,
            "dry_run": "false",
            "run_setup": "false"
        }
    )
    
    print("[Git] ✓ Git scan completed")

except Exception as e:
    logger.error(f"Git scan failed: {e}")
    raise